# Lezione 3 — Istogrammi e grafici a barre

Nella lezione precedente abbiamo raccontato storie **nel tempo** (linee, aree). Oggi rispondiamo a due domande diverse:

- Come si **distribuiscono** i valori? → **Istogramma**
- Come si **confrontano** le categorie con nome tra loro? → **Grafico a barre**

Continuiamo a lavorare con `df_can`, il dataset dell'immigrazione verso il Canada (1980-2013).

## 0. Preparazione: librerie e dati

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

%matplotlib inline
mpl.style.use('ggplot')

print('Matplotlib version:', mpl.__version__)

In [ ]:
df_can = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Canada.csv')

print('Dati caricati correttamente')
df_can.head()

In [ ]:
df_can.set_index('Country', inplace=True)

# lista degli anni 1980-2013 come testo, la useremo per tutto il corso
years = list(map(str, range(1980, 2014)))

print(df_can.shape)
df_can.head()

## 1. Istogrammi: organizzare la folla in gruppi

**Analogia:** immagina uno stadio pieno di gente e vuoi sapere come si distribuiscono le età. Non chiedi l'età esatta di ogni persona; le raggruppi in intervalli (0-10, 11-20, 21-30…) e conti quante persone cadono in ogni intervallo. Questo è un istogramma: un insieme di **bin** (cassetti) con la quantità di elementi che cadono in ciascuno.

Un istogramma **non** confronta categorie con nome (Canada vs Messico). Confronta **intervalli di un valore numerico**, per rivelare dove si concentra la maggior parte dei dati e dove ci sono valori estremi.

### 1.1 Istogramma semplice con `np.histogram`

In [ ]:
# np.histogram restituisce 2 valori: il conteggio per bin e i limiti di ogni bin
count, bin_edges = np.histogram(df_can['2013'])

print(count)      # quanti paesi cadono in ogni bin
print(bin_edges)  # i limiti di ogni bin (di default, 10 bin)

Con questo leggiamo la distribuzione in testo semplice: ad esempio, "178 paesi hanno contribuito tra 0 e 3412,9 immigrati nel 2013, 11 paesi tra 3412,9 e 6825,8", e così via. Pochi paesi (India, Cina) concentrano cifre enormi rispetto alla maggioranza.

In [ ]:
df_can['2013'].plot(kind='hist', figsize=(8, 5))

plt.title('Istogramma di immigrazione da 195 paesi nel 2013')
plt.ylabel('Numero di paesi')
plt.xlabel('Numero di immigrati')

plt.show()

Dettaglio fastidioso: i tick dell'asse x non coincidono con i limiti reali dei bin. Si risolve passando `bin_edges` come argomento `xticks`:

In [ ]:
count, bin_edges = np.histogram(df_can['2013'])

df_can['2013'].plot(kind='hist', figsize=(8, 5), xticks=bin_edges)

plt.title('Istogramma di immigrazione da 195 paesi nel 2013')
plt.ylabel('Numero di paesi')
plt.xlabel('Numero di immigrati')

plt.show()

### 1.2 Istogramma multi-serie: confrontare più distribuzioni contemporaneamente

Cosa succede se vogliamo confrontare la distribuzione di immigrati da **Danimarca, Norvegia e Svezia** tra il 1980 e il 2013? Il primo tentativo di solito fallisce:

```python
df_can.loc[['Denmark', 'Norway', 'Sweden'], years].plot.hist()
```

*pandas* interpreta ogni **colonna** del dataframe come una serie da graficare, e qui le colonne sono gli **anni**, non i paesi. Il risultato è un istogramma di anni, senza senso. La soluzione: **trasporre** il dataframe in modo che i paesi diventino colonne e gli anni righe.

In [ ]:
# trasporre affinché ogni paese sia una colonna
df_t = df_can.loc[['Denmark', 'Norway', 'Sweden'], years].transpose()

df_t.plot(kind='hist', figsize=(10, 6))

plt.title('Istogramma di immigrazione da Danimarca, Norvegia e Svezia (1980-2013)')
plt.ylabel('Numero di anni')
plt.xlabel('Numero di immigrati')

plt.show()

Ora affiniamo: più bin per più dettaglio, trasparenza per vedere le sovrapposizioni, colori propri per ogni serie.

In [ ]:
count, bin_edges = np.histogram(df_t, 15)

df_t.plot(kind='hist',
          figsize=(10, 6),
          bins=15,
          alpha=0.6,                                    # trasparenza: 0 = invisibile, 1 = opaco
          xticks=bin_edges,
          color=['coral', 'darkslateblue', 'mediumseagreen'])

plt.title('Istogramma di immigrazione da Danimarca, Norvegia e Svezia (1980-2013)')
plt.ylabel('Numero di anni')
plt.xlabel('Numero di immigrati')

plt.show()

Se preferiamo che le barre non si sovrappongano visivamente, le impiliamo con `stacked=True`:

In [ ]:
count, bin_edges = np.histogram(df_t, 15)
xmin = bin_edges[0] - 10   # margine estetico a sinistra
xmax = bin_edges[-1] + 10  # margine estetico a destra

df_t.plot(kind='hist',
          figsize=(10, 6),
          bins=15,
          xticks=bin_edges,
          color=['coral', 'darkslateblue', 'mediumseagreen'],
          stacked=True,
          xlim=(xmin, xmax))

plt.title('Istogramma di immigrazione da Danimarca, Norvegia e Svezia (1980-2013)')
plt.ylabel('Numero di anni')
plt.xlabel('Numero di immigrati')

plt.show()

## 2. Grafici a barre: confrontare categorie faccia a faccia

Un grafico a barre non distribuisce valori in intervalli: confronta **categorie con nome**. La lunghezza di ogni barra rappresenta la grandezza di quella categoria. Pensa a una classifica sportiva: ogni squadra è una barra, e l'altezza della barra è il suo punteggio. A colpo d'occhio vedi chi è primo e chi è ultimo.

In *pandas*, `kind='bar'` disegna barre **verticali** e `kind='barh'` disegna barre **orizzontali**.

### 2.1 Barre verticali: il caso dell'Islanda

Tra il 2008 e il 2011, l'Islanda subì il più grande collasso bancario relativo della storia economica moderna. Vediamo se questo si riflette nell'immigrazione islandese verso il Canada.

In [ ]:
df_iceland = df_can.loc['Iceland', years]

df_iceland.plot(kind='bar', figsize=(10, 6), rot=90)  # rot=90 ruota le etichette dell'asse x

plt.xlabel('Anno')
plt.ylabel('Numero di immigrati')
plt.title('Immigrati islandesi in Canada (1980-2013)')

plt.show()

Il salto dopo il 2008 è evidente. Possiamo segnalarlo con una freccia usando `plt.annotate`:

In [ ]:
df_iceland.plot(kind='bar', figsize=(10, 6), rot=90)

plt.xlabel('Anno')
plt.ylabel('Numero di immigrati')
plt.title('Immigrati islandesi in Canada (1980-2013)')

# la freccia: punta in xy, base in xytext
plt.annotate('',
             xy=(32, 70),        # punta della freccia: anno 2012, popolazione 70
             xytext=(28, 20),    # base della freccia: anno 2008, popolazione 20
             xycoords='data',    # usa il sistema di coordinate del grafico stesso
             arrowprops=dict(arrowstyle='->', connectionstyle='arc3', color='blue', lw=2))

# il testo che accompagna la freccia
plt.annotate('Crisi finanziaria 2008-2011',
             xy=(28, 30),
             rotation=72.5,   # angolo regolato a occhio per seguire la freccia
             va='bottom',     # allineamento verticale del testo
             ha='left')       # allineamento orizzontale del testo

plt.show()

`xy` è il punto di destinazione (la punta della freccia), `xytext` è il punto di origine (dove inizia il testo o la coda della freccia), e `arrowprops` controlla lo stile del tratto. Non serve memorizzare ogni parametro: si regolano per tentativi finché il risultato appare bene.

### 2.2 Barre orizzontali: quando il nome non ci sta

Quando le etichette sono lunghe (nomi di paesi), le barre verticali le schiacciano una contro l'altra. La soluzione: ruotare il grafico di 90 gradi con `kind='barh'`, che lascia molto più spazio sull'asse verticale per il testo.

Mostriamo i 15 paesi che hanno contribuito con più immigrati verso il Canada tra il 1980 e il 2013, con il valore totale annotato alla fine di ogni barra.

In [ ]:
# passo 1: ordinare e tenere i 15 maggiori
df_can.sort_values(by='Total', ascending=True, inplace=True)
df_top15 = df_can['Total'].tail(15)

# passo 2: creare il grafico
df_top15.plot(kind='barh', figsize=(12, 12), color='steelblue')
plt.xlabel('Numero di immigrati')
plt.title('I 15 paesi che hanno contribuito di più all\'immigrazione in Canada (1980-2013)')

# passo 3: annotare il valore di ogni barra
# enumerate() ci dà l'indice (posizione verticale) e il valore di ogni paese insieme
for index, value in enumerate(df_top15):
    label = format(int(value), ',')  # formatta il numero con virgole delle migliaia
    # collochiamo il testo vicino alla fine della barra, regolando manualmente lo spostamento
    plt.annotate(label, xy=(value - 47000, index - 0.10), color='white')

plt.show()

Il ciclo `enumerate(df_top15)` è l'elemento chiave: scorre i valori della serie e, allo stesso tempo, ti dà la posizione (`index`) di ciascuno, che corrisponde all'altezza di ogni barra orizzontale. Senza `enumerate` dovresti tenere il conto della posizione a mano.

## Punti chiave

- L'**istogramma** distribuisce un valore numerico in bin e conta quanti elementi cadono in ciascuno; risponde a "come si distribuiscono i miei dati?".
- Il **grafico a barre** confronta categorie con nome; risponde a "chi ha di più e chi ha di meno?".
- Se un istogramma di un dataframe viene male, sospetta prima dell'orientamento dei dati: probabilmente devi trasporre con `.transpose()`.
- `np.histogram()` ti dà i bin e i conteggi grezzi prima di disegnare qualsiasi cosa; usalo per passare `bin_edges` come `xticks`.
- `kind='bar'` è verticale, `kind='barh'` è orizzontale; usa le barre orizzontali quando le etichette delle categorie sono lunghe.
- `enumerate()` dentro un ciclo ti dà indice e valore insieme, ideale per annotare ogni barra con `plt.annotate`.

## Prossimo passo

Apri `clase03_practica_histogramas-barras_IT.ipynb` e risolvi gli esercizi proposti.